In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/diabetes.csv")

print("Shape:", df.shape)
df.head()

Shape: (768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [2]:
import numpy as np

# Columns where a true zero is medically impossible
zero_invalid_cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

print("Count of zero values in each column (likely = missing data):")
for col in zero_invalid_cols:
    zero_count = (df[col] == 0).sum()
    pct = (zero_count / len(df)) * 100
    print(f"{col}: {zero_count} zeros ({pct:.1f}%)")

print("\nBasic info:")
df.info()

print("\nTrue missing values (NaN) per column:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nOutcome class distribution:")
print(df["Outcome"].value_counts())
print(df["Outcome"].value_counts(normalize=True) * 100)

Count of zero values in each column (likely = missing data):
Glucose: 5 zeros (0.7%)
BloodPressure: 35 zeros (4.6%)
SkinThickness: 227 zeros (29.6%)
Insulin: 374 zeros (48.7%)
BMI: 11 zeros (1.4%)

Basic info:
<class 'pandas.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB

True missing values (NaN) per column:
Pr

In [3]:
# Step 1: Replace biologically impossible zeros with NaN,
# so pandas treats them as genuinely missing (not real measurements)
cols_with_disguised_missing = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

df_clean = df.copy()
for col in cols_with_disguised_missing:
    df_clean[col] = df_clean[col].replace(0, np.nan)

print("Missing values AFTER converting zeros to NaN:")
print(df_clean[cols_with_disguised_missing].isnull().sum())

# Step 2: Impute missing values using the MEDIAN of each column
# (median is robust to outliers, unlike mean)
for col in cols_with_disguised_missing:
    median_value = df_clean[col].median()
    df_clean[col] = df_clean[col].fillna(median_value)
    print(f"{col}: filled {df[col].eq(0).sum()} missing values with median = {median_value}")

print("\nMissing values AFTER imputation (should all be 0):")
print(df_clean.isnull().sum())

# Step 3: Save the cleaned dataset
df_clean.to_csv("../data/processed/diabetes_clean.csv", index=False)
print("\nCleaned dataset saved to ml/data/processed/diabetes_clean.csv")

Missing values AFTER converting zeros to NaN:
Glucose            5
BloodPressure     35
SkinThickness    227
Insulin          374
BMI               11
dtype: int64
Glucose: filled 5 missing values with median = 117.0
BloodPressure: filled 35 missing values with median = 72.0
SkinThickness: filled 227 missing values with median = 29.0
Insulin: filled 374 missing values with median = 125.0
BMI: filled 11 missing values with median = 32.3

Missing values AFTER imputation (should all be 0):
Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

Cleaned dataset saved to ml/data/processed/diabetes_clean.csv


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Separate features and target
X = df_clean.drop("Outcome", axis=1)
y = df_clean["Outcome"]

# 80:20 stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

# Scale features (fit on training data only)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Compare all 4 required algorithms
models = {
    "Random Forest": RandomForestClassifier(random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "K-Nearest Neighbors": KNeighborsClassifier(),
    "Support Vector Machine": SVC(random_state=42, probability=True)
}

results = []
for name, model in models.items():
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=10, scoring="accuracy")
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    results.append({
        "Model": name,
        "CV Mean Accuracy": cv_scores.mean(),
        "CV Std": cv_scores.std(),
        "Test Accuracy": accuracy_score(y_test, y_pred),
        "Test Precision": precision_score(y_test, y_pred),
        "Test Recall": recall_score(y_test, y_pred),
        "Test F1": f1_score(y_test, y_pred)
    })

results_df = pd.DataFrame(results).sort_values("CV Mean Accuracy", ascending=False)
results_df

Training set: (614, 8)
Testing set: (154, 8)


D:\ai-disease-detection\venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
D:\ai-disease-detection\venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
D:\ai-disease-detection\venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
D:\ai-disease-detection\venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in ve

,Model,CV Mean Accuracy,CV Std,Test Accuracy,Test Precision,Test Recall,Test F1
3,Support Vector Machine,0.772078,0.026375,0.740260,0.652174,0.555556,0.600000
0,Random Forest,0.767134,0.046929,0.779221,0.717391,0.611111,0.660000
2,K-Nearest Neighbors,0.737943,0.052854,0.753247,0.660000,0.611111,0.634615
1,Decision Tree,0.659625,0.063102,0.681818,0.553191,0.481481,0.514851


In [6]:
import joblib

best_model = models["Random Forest"]

joblib.dump(best_model, "../saved_models/diabetes_model.pkl")
joblib.dump(scaler, "../saved_models/diabetes_scaler.pkl")

print("Diabetes model and scaler saved successfully.")

Diabetes model and scaler saved successfully.


In [7]:
positive_idx = y_test[y_test == 1].index[0]
negative_idx = y_test[y_test == 0].index[0]

print("=== CONFIRMED POSITIVE CASE (Outcome=1) ===")
print(X_test.loc[positive_idx].to_dict())

print("\n=== CONFIRMED NEGATIVE CASE (Outcome=0) ===")
print(X_test.loc[negative_idx].to_dict())

=== CONFIRMED POSITIVE CASE (Outcome=1) ===
{'Pregnancies': 7.0, 'Glucose': 114.0, 'BloodPressure': 64.0, 'SkinThickness': 29.0, 'Insulin': 125.0, 'BMI': 27.4, 'DiabetesPedigreeFunction': 0.732, 'Age': 34.0}

=== CONFIRMED NEGATIVE CASE (Outcome=0) ===
{'Pregnancies': 7.0, 'Glucose': 159.0, 'BloodPressure': 64.0, 'SkinThickness': 29.0, 'Insulin': 125.0, 'BMI': 27.4, 'DiabetesPedigreeFunction': 0.294, 'Age': 40.0}


In [8]:
# Scan through several test patients and find ones the model gets RIGHT,
# so we have a reliable, correct case for live demonstration
sample_results = []
for idx in X_test.index[:20]:
    row = X_test.loc[idx]
    true_label = y_test.loc[idx]
    pred = best_model.predict(scaler.transform(row.values.reshape(1, -1)))[0]
    correct = "✓" if pred == true_label else "✗"
    sample_results.append({"index": idx, "true": true_label, "predicted": pred, "correct": correct})

results_check = pd.DataFrame(sample_results)
print(results_check)

# Show full details of the first CORRECTLY predicted positive case
correct_positive_idx = results_check[(results_check["true"] == 1) & (results_check["correct"] == "✓")].iloc[0]["index"]
print("\n=== RELIABLE DEMO CASE (correctly predicted, Outcome=1) ===")
print(X_test.loc[correct_positive_idx].to_dict())

D:\ai-disease-detection\venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
D:\ai-disease-detection\venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
D:\ai-disease-detection\venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
D:\ai-disease-detection\venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
D:\ai-disease-detection\venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
D:\ai-disease-d

    index  true  predicted correct
0      44     0          1       ✗
1     672     0          0       ✓
2     700     0          0       ✓
3     630     1          0       ✗
4      81     0          0       ✓
5     389     0          0       ✓
6     387     1          1       ✓
7     408     1          1       ✓
8     163     0          0       ✓
9     335     0          1       ✗
10    471     0          0       ✓
11     78     1          1       ✓
12    307     0          0       ✓
13    392     0          0       ✓
14    678     1          0       ✗
15    116     1          0       ✗
16    660     0          1       ✗
17    742     0          0       ✓
18    260     0          1       ✗
19    533     0          0       ✓

=== RELIABLE DEMO CASE (correctly predicted, Outcome=1) ===
{'Pregnancies': 8.0, 'Glucose': 105.0, 'BloodPressure': 100.0, 'SkinThickness': 36.0, 'Insulin': 125.0, 'BMI': 43.3, 'DiabetesPedigreeFunction': 0.239, 'Age': 45.0}


D:\ai-disease-detection\venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
D:\ai-disease-detection\venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
D:\ai-disease-detection\venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
D:\ai-disease-detection\venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
D:\ai-disease-detection\venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [9]:
correct_negative_idx = results_check[(results_check["true"] == 0) & (results_check["correct"] == "✓")].iloc[0]["index"]
print("=== RELIABLE DEMO CASE (correctly predicted, Outcome=0) ===")
print(X_test.loc[correct_negative_idx].to_dict())

=== RELIABLE DEMO CASE (correctly predicted, Outcome=0) ===
{'Pregnancies': 10.0, 'Glucose': 68.0, 'BloodPressure': 106.0, 'SkinThickness': 23.0, 'Insulin': 49.0, 'BMI': 35.5, 'DiabetesPedigreeFunction': 0.285, 'Age': 47.0}
